# Session 3: FastAPI 기초

## FastAPI 소개

### FastAPI란?

FastAPI는 Python으로 API를 만들기 위한 **현대적이고 빠른 웹 프레임워크**입니다.

### 왜 FastAPI인가?

| 항목 | Flask | FastAPI |
|------|-------|---------|
| 속도 | 보통 | 매우 빠름 (async 지원) |
| 타입 검증 | 수동 구현 필요 | Pydantic으로 자동 |
| API 문서 | 별도 설정 필요 | Swagger UI 자동 생성 |
| 학습 곡선 | 낮음 | 낮음 |
| ML 서빙 적합도 | 보통 | 매우 높음 |

### FastAPI의 핵심 장점

1. **자동 문서화**: 코드만 작성하면 Swagger UI(/docs)가 자동으로 생성됩니다.
2. **타입 검증**: Pydantic 모델로 입력 데이터의 타입과 유효성을 자동 검증합니다.
3. **비동기 지원**: async/await로 높은 동시 처리 성능을 제공합니다.
4. **ML 모델 서빙에 최적**: 모델 로드 → 입력 검증 → 예측 → 응답의 흐름이 자연스럽습니다.

In [ ]:
!pip install fastapi uvicorn

## 최소 FastAPI 앱 만들기

가장 간단한 FastAPI 애플리케이션을 만들어봅시다.  
`%%writefile` 매직 커맨드를 사용하면 셀의 내용을 파일로 저장할 수 있습니다.

### 기본 구조

```python
from fastapi import FastAPI

app = FastAPI()  # FastAPI 인스턴스 생성

@app.get("/")    # GET 요청을 처리하는 엔드포인트
def root():
    return {"message": "Hello"}  # JSON 응답 자동 변환
```

In [ ]:
# 여기에 코드를 작성하세요


## 서버 실행 방법

터미널에서 다음 명령어로 서버를 실행합니다:

```bash
uvicorn main:app --reload
```

### 명령어 해석

| 부분 | 의미 |
|------|------|
| `uvicorn` | ASGI 서버 (FastAPI를 실행하는 서버) |
| `main` | 파일명 (main.py) |
| `app` | FastAPI 인스턴스 변수명 |
| `--reload` | 코드 변경 시 서버 자동 재시작 (개발 모드) |

### 추가 옵션

```bash
# 포트 변경 (기본 8000)
uvicorn main:app --reload --port 8080

# 외부 접속 허용 (0.0.0.0)
uvicorn main:app --reload --host 0.0.0.0
```

서버 실행 후 브라우저에서 `http://localhost:8000` 으로 접속할 수 있습니다.

## Swagger UI 활용법

FastAPI는 **API 문서를 자동으로 생성**해줍니다. 이것이 FastAPI의 가장 큰 장점 중 하나입니다.

### Swagger UI (`/docs`)

- 주소: `http://localhost:8000/docs`
- 각 엔드포인트를 **직접 테스트**할 수 있는 인터랙티브 UI
- "Try it out" 버튼으로 바로 API 호출 가능
- 요청/응답 형식, 파라미터 정보를 한눈에 확인

### ReDoc (`/redoc`)

- 주소: `http://localhost:8000/redoc`
- 더 깔끔한 형태의 API 문서
- 문서 공유/인쇄에 적합한 레이아웃

### 왜 자동 문서화가 중요한가?

- 프론트엔드 개발자에게 API 스펙을 별도로 작성할 필요가 없습니다.
- 코드가 곧 문서이므로 문서와 코드 불일치 문제가 사라집니다.
- QA 팀이 직접 API를 테스트할 수 있습니다.

## Path Parameter와 Query Parameter

### Path Parameter (경로 매개변수)

URL 경로의 일부로 값을 전달합니다:
```
GET /users/123      → user_id = 123
GET /items/apple    → item_name = "apple"
```

### Query Parameter (쿼리 매개변수)

URL 뒤에 `?key=value` 형태로 값을 전달합니다:
```
GET /items?skip=0&limit=10
GET /search?q=python&sort=date
```

### 언제 무엇을 사용하나?

- **Path Parameter**: 리소스를 식별할 때 (필수 값) → `/users/{user_id}`
- **Query Parameter**: 필터링, 정렬, 페이징 등 선택적 옵션 → `?sort=asc&limit=10`

In [ ]:
# 여기에 코드를 작성하세요


## POST 요청과 Pydantic

### GET vs POST

| 항목 | GET | POST |
|------|-----|------|
| 용도 | 데이터 조회 | 데이터 생성/전송 |
| 데이터 전달 | URL에 포함 | Request Body에 포함 |
| 데이터 크기 | URL 길이 제한 | 제한 없음 |
| ML 예측 | 부적합 | **적합** (피처 데이터 전송) |

### Pydantic이란?

Pydantic은 **데이터 검증 라이브러리**입니다. FastAPI에서 핵심적인 역할을 합니다:

1. **입력 데이터 타입 검증**: 문자열이 와야 할 곳에 숫자가 오면 자동 에러
2. **자동 변환**: 가능한 경우 타입을 자동 변환 ("123" → 123)
3. **문서 자동 생성**: Swagger UI에 요청 형식이 자동으로 표시됨
4. **기본값 설정**: 선택적 필드에 기본값 지정 가능

### ML 서빙에서 Pydantic의 역할

```
클라이언트 → [Pydantic 검증] → [모델 예측] → [응답]
              타입, 범위 확인     전처리+추론    JSON 반환
```

잘못된 입력이 모델에 전달되면 에러나 잘못된 예측이 발생합니다.  
Pydantic으로 **모델 호출 전에** 입력을 검증하여 이를 방지합니다.

In [ ]:
# 여기에 코드를 작성하세요


## 응답 모델과 자동 문서화

### response_model의 장점

```python
@app.post("/predict", response_model=PredictionResponse)
```

1. **응답 필터링**: 모델에 정의된 필드만 응답에 포함됩니다. (민감 정보 유출 방지)
2. **타입 변환**: 응답 데이터의 타입이 자동으로 변환됩니다.
3. **문서화**: Swagger UI에 응답 형식이 명확하게 표시됩니다.

### ML 예측 API에서의 활용

```python
class PredictionRequest(BaseModel):
    age: int
    income: float
    credit_score: int
    # ... 모든 피처

class PredictionResponse(BaseModel):
    prediction: int          # 0 또는 1
    probability: float       # 승인 확률
    label: str              # "승인" 또는 "거절"
```

이 구조를 Session 4에서 실제 대출 예측 API에 적용할 예정입니다.

In [ ]:
# 여기에 코드를 작성하세요


## 정리

### 이번 세션에서 배운 것

1. **FastAPI 기본 구조**: `FastAPI()` 인스턴스 생성, 데코레이터로 엔드포인트 정의
2. **GET 요청**: `@app.get("/path")` - 데이터 조회용
3. **Path/Query Parameter**: URL을 통한 데이터 전달 방법
4. **POST 요청**: `@app.post("/path")` - 데이터 전송용 (ML 예측에 사용)
5. **Pydantic 모델**: 입력 데이터 타입 검증 + 자동 문서화
6. **Swagger UI**: `/docs`에서 API를 바로 테스트
7. **서버 실행**: `uvicorn main:app --reload`

### 핵심 파일

| 파일 | 내용 |
|------|------|
| `main.py` | 기본 GET 엔드포인트 (루트, 헬스체크) |
| `main_params.py` | Path/Query Parameter 예제 |
| `main_post.py` | POST + Pydantic 모델 예제 |

